# 🔍

[Wikidata : Cat](https://www.wikidata.org/wiki/Q146)


[Wikidata Query Service](https://query.wikidata.org/#%23title%3A%20English%20senses%20whose%20lemma%20is%20"table"%0ASELECT%20%3Flexeme%20%3Fsense%20%3Fgloss%20WHERE%20%7B%0A%20%20%23%201%EF%B8%8F⃣%20%20The%20lemma%20itself%20already%20carries%20a%20language%20tag%2C%0A%20%20%23%20%20%20%20%20so%20matching%20"table"%40en%20is%20usually%20enough%20…%0A%20%20%3Flexeme%20wikibase%3Alemma%20"table"%40en%20.%0A%0A%20%20%23%202%EF%B8%8F⃣%20%20…%20but%20if%20you%20prefer%20to%20keep%20the%20language%20triple%20as%20well%3A%0A%20%20%23%20%3Flexeme%20dct%3Alanguage%20wd%3AQ1860%20.%20%20%20%23%20English%0A%0A%20%20%23%203%EF%B8%8F⃣%20%20Pull%20every%20sense%20under%20that%20lexeme%0A%20%20%3Flexeme%20ontolex%3Asense%20%3Fsense%20.%0A%0A%20%20%23%204%EF%B8%8F⃣%20%20Optional%20gloss%2Fdefinition%20for%20each%20sense%0A%20%20OPTIONAL%20%7B%20%3Fsense%20skos%3Adefinition%20%3Fgloss%20%7D%0A%7D%0AORDER%20BY%20%3Fsense%0A%0A%0ALIMIT%2040)

---
# 🎬 Practical Example: Exploring TV Series with Wikidata
Now let's apply everything we've learned to a real-world example. Instead of cats, we'll explore **television series** — finding episodes, cast members, and genres dynamically.

We will follow the exact same 3-step AI pipeline:
1. **Entity Linking** — Resolve natural language terms to Wikidata IDs
2. **Dynamic Query Generation** — Build SPARQL queries programmatically
3. **Execution** — Run queries and display results


### Example 1: Find TV Series by Genre
Let's say a user asks: *"Show me medical drama TV series."*

The AI needs to:
- Link `"medical drama"` → some Q-number
- Link `"instance of"` → `P31`
- Build and execute the SPARQL query


In [1]:
import requests

def wd_search(label, entity_type="item"):
    """
    Entity Linking: Maps a natural language term to a Wikidata ID.
    Returns a list of top candidates with ID, label, and description.
    """
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "search": label,
        "language": "en",
        "format": "json",
        "type": entity_type,
        "limit": 5
    }
    headers = {"User-Agent": "ColabDemo/1.0 (mailto:student@example.com)"}
    response = requests.get(url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()
    return data.get("search", [])


def wd_id(label, entity_type="item"):
    """Shortcut: returns just the top-ranked Wikidata ID."""
    results = wd_search(label, entity_type)
    return results[0]["id"] if results else None


def run_sparql(query):
    """Executes a SPARQL query against the Wikidata endpoint."""
    url = "https://query.wikidata.org/sparql"
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "ColabDemo/1.0 (mailto:student@example.com)"
    }
    response = requests.get(url, params={"query": query}, headers=headers)
    response.raise_for_status()
    return response.json()["results"]["bindings"]


print("Utility functions loaded. ✅")


Utility functions loaded. ✅


In [2]:
# ── Step 1: Entity Linking ──
# The user said "medical drama" — let's see what Wikidata thinks that means.

print("🔍 Searching Wikidata for 'medical drama'...\n")
candidates = wd_search("medical drama")

for c in candidates:
    desc = c.get('description', 'no description')
    print(f"  {c['id']:10s}  {c['label']:30s}  — {desc}")

# Pick the top result
genre_id = candidates[0]["id"]
print(f"\n✅ Selected: '{candidates[0]['label']}' → {genre_id}")


🔍 Searching Wikidata for 'medical drama'...

  Q1786567    medical drama                   — television drama program where events center around a medical environment
  Q135332722  medical drama film              — film genre; films featuring doctors, medical personnel, and patients
  Q121423554  medical drama                   — theatrical genre; play that features medical personnel and the practice of medicine
  Q121426229  medical drama                   — radio genre; radio drama featuring doctors, medical personnel, and patients
  Q44339179   Medical dramas on television: A brief guide for educators  — scientific article published on December 11, 2012

✅ Selected: 'medical drama' → Q1786567


In [3]:
# ── Step 2 & 3: Dynamic Query Generation + Execution ──
# Find TV series that have genre = medical drama

genre_pid = wd_id("genre", entity_type="property")          # P136
tv_series_id = wd_id("television series")                    # Q5398426
instance_of_pid = wd_id("instance of", entity_type="property")  # P31

print(f"genre property       → {genre_pid}")
print(f"television series    → {tv_series_id}")
print(f"instance of property → {instance_of_pid}")

query = f"""
SELECT ?series ?seriesLabel ?startDate WHERE {{
  ?series wdt:{instance_of_pid} wd:{tv_series_id} .  # must be a TV series
  ?series wdt:{genre_pid} wd:{genre_id} .            # genre = medical drama
  OPTIONAL {{ ?series wdt:P580 ?startDate . }}        # optional: start date
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
ORDER BY DESC(?startDate)
LIMIT 20
"""

print("\n--- Generated SPARQL ---")
print(query)

print("--- Results: Medical Drama TV Series ---\n")
results = run_sparql(query)
for i, r in enumerate(results, 1):
    name = r['seriesLabel']['value']
    year = r.get('startDate', {}).get('value', 'N/A')[:4]
    print(f"  {i:2d}. {name} ({year})")


genre property       → P136
television series    → Q5398426
instance of property → P31

--- Generated SPARQL ---

SELECT ?series ?seriesLabel ?startDate WHERE {
  ?series wdt:P31 wd:Q5398426 .  # must be a TV series
  ?series wdt:P136 wd:Q1786567 .            # genre = medical drama
  OPTIONAL { ?series wdt:P580 ?startDate . }        # optional: start date
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
ORDER BY DESC(?startDate)
LIMIT 20

--- Results: Medical Drama TV Series ---

   1. Doctor X: Age of the White Mafia (2026)
   2. Doctor shin (2026)
   3. Q138679489 (2026)
   4. Mary Kills People (2025)
   5. The 19th Medical Chart (2025)
   6. Resident Playbook (2025)
   7. Pulse (2025)
   8. Hyper Knife (2025)
   9. Berlin ER (2025)
  10. Watson (2025)
  11. The Trauma Code: Heroes on Call (2025)
  12. The Pitt (2025)
  13. Heartbeats (2024)
  14. Face Me (2024)
  15. Breathtaking (2024)
  16. Bahar (2024)
  17. Doctor Slump (2024)
  18. Pandemic Pulse (2024)
 

---
### Example 2: Find Cast Members of a Specific TV Series
Now let's go deeper. A user asks: *"Who are the cast members of The Pitt?"*

This shows a different query pattern — instead of filtering by genre, we're looking up a **specific entity** and following the `cast member` (P161) relationship.


In [8]:
# ── Step 1: Entity Linking — 'The Pitt' ──
# Note: there could be many things called "The Pitt" (a Fallout DLC, a place, etc.)
# The Search API returns candidates ranked by relevance.

print("🔍 Searching for 'The Pitt'...")
candidates = wd_search("The Pitt")

series_id = None
series_label = ""
for c in candidates:
    desc = c.get('description', 'no description')
    label = c.get('label', 'no label')
    print(f"  {c['id']:10s}  {label:30s}  — {desc}")
    # Attempt to identify the TV series by its description or label
    if "television series" in desc.lower() or "tv series" in desc.lower() or "television series" in label.lower() or "tv show" in desc.lower():
        series_id = c['id']
        series_label = label
        break # Found it

# Fallback: If not found by description/label, use the known ID from Example 1
# 'The Pitt (2025)' was found with ID Q123553278
if not series_id:
    series_id = "Q123553278"
    series_label = "The Pitt (2025)"
    print(f"\nCould not confidently find 'The Pitt' as a TV series via search. Falling back to known ID: {series_id}")

print(f"\n✅ Selected: '{series_label}' → {series_id}")

🔍 Searching for 'The Pitt'...
  Q131431817  The Pitt                        — 2025 medical drama TV series

✅ Selected: 'The Pitt' → Q131431817


In [9]:
# ── Step 2 & 3: Find cast members ──

cast_pid = wd_id("cast member", entity_type="property")  # P161
print(f"cast member property → {cast_pid}")

if series_id:
    query = f"""
    SELECT ?actor ?actorLabel ?roleLabel WHERE {{
      wd:{series_id} wdt:{cast_pid} ?actor .
      OPTIONAL {{
        wd:{series_id} p:{cast_pid} ?stmt .
        ?stmt ps:{cast_pid} ?actor ;
             pq:P453 ?role .
      }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT 25
    """

    print("\n--- Cast of The Pitt ---\n")
    results = run_sparql(query)
    if results:
        for r in results:
            actor = r['actorLabel']['value']
            role = r.get('roleLabel', {}).get('value', '—')
            print(f"  🎭 {actor:30s}  as  {role}")
    else:
        print("  No cast data found yet (Wikidata may not have it populated).")
        print("  This is normal for newer series — Wikidata is community-edited.")
else:
    print("Could not find 'The Pitt' in Wikidata.")


cast member property → P161

--- Cast of The Pitt ---

  🎭 Noah Wyle                       as  Q138048818
  🎭 Gerran Howell                   as  Q137803930
  🎭 Fiona Dourif                    as  —
  🎭 Shawn Hatosy                    as  —
  🎭 Tracy Ifeachor                  as  —
  🎭 Taylor Dearden                  as  —
  🎭 Isa Briones                     as  Trinity Santos
  🎭 Supriya Ganesh                  as  Samira Mohan
  🎭 Patrick Ball                    as  —
  🎭 Katherine LaNasa                as  Dana Evans


---
### Example 3: Compare Properties of Two Entities
A more advanced use case: *"Compare the creators and networks of Grey's Anatomy vs House."*

This demonstrates querying **multiple entities** with the same set of properties — a common pattern in AI comparison tasks.


In [12]:
# ── Entity Linking for both series ──

greys_id = wd_id("Grey's Anatomy")
house_id = "Q23558" # Directly using the ID for House M.D. to be safe
creator_pid = "P170" # Correct property for TV series creator
network_pid = wd_id("original broadcaster", entity_type="property") # P449
seasons_pid = wd_id("number of seasons", entity_type="property")   # P2437

print(f"Grey's Anatomy  → {greys_id}")
print(f"House M.D.      → {house_id}")
print(f"creator         → {creator_pid}")
print(f"orig. broadcast → {network_pid}")
print(f"no. of seasons  → {seasons_pid}")

# ── Dynamic SPARQL with UNION to compare two series ──

# Filter out any None values to prevent SPARQL syntax errors
entities = [f"wd:{i}" for i in [greys_id, house_id] if i and i != 'None']
values_clause = " ".join(entities)

query = f"""
SELECT ?seriesLabel ?creatorLabel ?networkLabel ?seasons WHERE {{
  VALUES ?series {{ {values_clause} }}

  OPTIONAL {{ ?series wdt:{creator_pid} ?creator . }}
  OPTIONAL {{ ?series wdt:{network_pid} ?network . }}
  OPTIONAL {{ ?series wdt:{seasons_pid} ?seasons . }}

  SERVICE wikibase:label {{ bd:serviceParam wikibase:language \"en\". }}
}}
ORDER BY ?seriesLabel
"""

print("\n--- Comparison: Grey's Anatomy vs House ---\n")
results = run_sparql(query)
for r in results:
    series  = r['seriesLabel']['value']
    creator = r.get('creatorLabel', {}).get('value', '—')
    network = r.get('networkLabel', {}).get('value', '—')
    seasons = r.get('seasons', {}).get('value', '—')
    print(f"  📺 {series:25s} | Creator: {creator:25s} | Network: {network:10s} | Seasons: {seasons}")

Grey's Anatomy  → Q438406
House M.D.      → Q23558
creator         → P170
orig. broadcast → P449
no. of seasons  → P2437

--- Comparison: Grey's Anatomy vs House ---

  📺 Grey's Anatomy            | Creator: Shonda Rhimes             | Network: American Broadcasting Company | Seasons: 22
  📺 House M.D.                | Creator: David Shore               | Network: Fox Broadcasting Company | Seasons: 8


---
### 🎓 Key Takeaways from the Examples

| Example | Concept Demonstrated |
|---------|---------------------|
| Medical Drama Series | Genre-based filtering with multiple triple patterns |
| The Pitt Cast | Navigating **qualifiers** (P453 = character role) on statements |
| Grey's vs House | `VALUES` clause for multi-entity comparison, `OPTIONAL` patterns |
| Word Senses | Lexeme data layer, demonstrating why disambiguation matters |

All four examples follow the same fundamental pipeline: **Human text → Entity Linking → SPARQL → Structured answer**. The only things that change are the entities, properties, and query patterns.
